In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
arr = np.array(Image.open("me.png"))

In [ ]:
arr = arr[20:920, :900, :]
arr = arr.min(axis=-1)
arr = 255 - arr

In [ ]:
plt.hist(arr)

In [ ]:
Image.fromarray(arr)

In [ ]:
(arr < 125).sum() / (arr > 125).sum() 

In [ ]:
import torch.nn.functional as F

K = 20
# net = nn.Linear(2, 2*K, bias=True)
net = nn.Sequential(
    nn.Linear(2, 2*K, bias=True),
    nn.ReLU(),
    nn.Linear(2*K, 1, bias=False)
)
with torch.no_grad():
    net[0].weight.mul_(0.1)
nn.init.normal_(net[0].bias, mean=0.0, std=0.1)
alpha = torch.tensor(1.0, requires_grad=True)

optimizer = torch.optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-2)

x = torch.linspace(-1, 1, 900)
y = torch.linspace(-1, 1, 900)
xx, yy = torch.meshgrid(x, y)

data = torch.stack([xx, yy], axis=2)
truth = torch.tensor(arr) / 255
data = data.view(-1, 2)
truth = truth.view(-1)
# truth += abs(torch.randn_like(truth) / 100)

HW = data.shape[0]
batch = 2048

In [ ]:
for i in range(1000):
    idx = torch.randint(HW, size=(batch,))
    xy = data[idx]
    y_true = truth[idx]
    # print(torch.sum(y_true>1/2))
    mask = (y_true < 1/2) * 1/6 + (y_true > 1/2) * 1
    # z = net(xy)
    # z_pos = z[..., :K]
    # z_neg = z[..., K:]
    # pred = alpha * F.leaky_relu(z_pos).sum(axis=-1) + F.leaky_relu(z_neg).sum(axis=-1)
    pred = net(xy)
    loss = (mask * (pred - y_true)**2).mean()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if i % 10 == 0:
        print(loss.item())

In [ ]:
pred = net(data)
# pred = alpha * F.leaky_relu(z_pos).sum(axis=-1) + F.leaky_relu(z_neg).sum(axis=-1)

In [ ]:
import matplotlib.pyplot as plt

_ = plt.hist(pred.detach().numpy())

In [ ]:
pred.mean()

In [ ]:
Image.fromarray((pred.clamp(0, 1)*255).to(torch.uint8).view(900, 900).numpy(), mode="L")

In [ ]:
division = []
with open("activation.txt", "w") as f:
    f.write("[\n")
    for (a, b), c, mul in zip(net[0].weight.tolist(), net[0].bias.tolist(), net[2].weight[0].tolist()):
        a *= mul
        b *= mul
        c *= mul
        f.write(f"({a}, {b}, {c}),\n")
        division.append((a, b, c))
    f.write("]")

In [ ]:
from visualize import *
plot_relu_partition(division)

In [ ]:
render_relu(division)

In [ ]:
how_many_closed_sub_space(division)